In [8]:
import pyarrow as pa
import pyarrow.parquet as pq
import os
import yaml

In [9]:
#Load the configuration file
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

raw_data_path = "../"+config["paths"]["raw_data_dir"]    

## Sampling the data

I will sample the data to make it light in resources, as the original dataset is fairly large.

In [3]:
# Retrieve the path and file names for all the parquet files

def get_parquet_from_path(path):
    parquet_files = []
    for (dir_path, _, files) in os.walk(path):
        for f in files:
            if f.endswith(".parquet"):
                parquet_files.append(os.path.join(dir_path, f)) 
    return parquet_files

In [4]:
# Check the method get_parquet_from_path() output

file_path = get_parquet_from_path(raw_data_path)
(print(file_path))


['../data/raw/cscareerquestionsEU_comments.parquet', '../data/raw/SoftwareEngineering_comments.parquet', '../data/raw/cscareerquestions_comments.parquet', '../data/raw/SoftwareEngineering_submissions.parquet', '../data/raw/datascience_submissions.parquet', '../data/raw/cybersecurity_submissions.parquet', '../data/raw/QualityAssurance_comments.parquet', '../data/raw/datascience_comments.parquet', '../data/raw/QualityAssurance_submissions.parquet', '../data/raw/cscareerquestionsEU_submissions.parquet', '../data/raw/cscareerquestions_submissions.parquet', '../data/raw/dataengineering_submissions.parquet', '../data/raw/cybersecurity_comments.parquet', '../data/raw/dataengineering_comments.parquet']


### Core Sampler

In [ ]:
import json
import random
def sample_parquet_file_to_json(file_path, desired_samples = 10000):
    file_count = 0
    for file_number in range(len(file_path)):
        file_name = os.path.splitext(os.path.basename(file_path[file_number]))[0]
        # Read the paraquet file metadata
        metadata = pq.read_metadata(file_path[file_number])
        # Total number of samples and target retention number
        total_samples = metadata.num_rows
        # Perform random sampling
        if total_samples <= desired_samples:
            table = pq.read_table(file_path[file_number])
            data = table.to_pylist()
        else:
            random_indices = random.sample(range(total_samples), desired_samples)
            random_indices.sort()  # Sort the indices to speed up batch reading
            table = pq.read_table(file_path[file_number])
          
            # Take a batch based on the random  indices
            sampled_table = table.take(random_indices)
            selected_samples = sampled_table.to_pylist()
        # Write the sampled results to a new JSON file
        output_dir_path = ("../data/samples")
        os.makedirs(output_dir_path, exist_ok=True)
        with open(f'{output_dir_path}/{file_name}.json','w', encoding='utf-8') as f:
            json.dump(selected_samples, f, ensure_ascii=False, indent=4, default=str)
        print(f"Random sampling completed for {file_name}, retained {len(selected_samples)} samples.") 
        file_count+=1
    print(f"Done! Sampled {file_count} files")   

In [6]:
#sample_parquet_file_to_json(file_path)

In [ ]:
data_samples_path = "../"+config["paths"]["data_samples_dir"]

'../data/samples'